# 🎯 Alzheimer's Disease Classification - Inference Demo
## Multi-Orientation CNN-DeiT Hybrid with Cross-Attention

**Purpose**: Quick demonstration notebook for thesis viva/review presentations

**What this notebook does:**
1. ✅ Loads pre-trained Phase 3 ADVANCED model
2. ✅ Takes any test MRI image (3 orientations)
3. ✅ Shows comprehensive prediction results with visualizations
4. ✅ Displays confidence scores and model interpretability
5. ✅ Runs in ~1-2 minutes (no training required!)

**Use Case**: Show reviewers/examiners how the trained model makes predictions on new data

In [ ]:
# Install required packages (if needed)
!pip install timm -q
print("✅ Dependencies ready")

In [ ]:
import os
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F

# Torchvision
from torchvision import transforms, models

# Timm for DeiT
import timm

warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️  Device: {device}")
if torch.cuda.is_available():
    print(f"🎮 GPU: {torch.cuda.get_device_name(0)}")
    print(f"💾 GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
else:
    print("⚠️  Running on CPU (slower)")

print("\n✅ All libraries imported!")

## 📂 Configuration

**Update these paths to match your setup:**
- Model checkpoint from Phase 3 training
- Test data with 3 orientations
- Phase 2 CNN feature extractors

In [ ]:
# ============================================================================
# PATHS - UPDATE THESE FOR YOUR ENVIRONMENT
# ============================================================================

# Trained model checkpoint (from Phase 3)
MODEL_CHECKPOINT = "/kaggle/working/phase3_advanced/advanced_cnn_vit_best.pth"
# Or local: "e:/MTech ASE/.../phase3_advanced_outputs/advanced_cnn_vit_best.pth"

# Test data
DATA_PATH = "/kaggle/input/adni-preprocessed-5class/ADNI_PHASE1_PROCESSED"
# Or local: "e:/MTech ASE/.../ADNI_PHASE1_PROCESSED"

# Phase 2 CNN extractors (frozen features)
PHASE2_OUTPUT = "/kaggle/input/phase2-outputs"
# Or local: "e:/MTech ASE/.../Dataset & Outputs/phase2_outputs"

# Test splits CSV
TEST_CSV = "/kaggle/input/phase1-outputs/test_balanced.csv"
# Or local: "e:/MTech ASE/.../Dataset & Outputs/phase1_outputs/test_balanced.csv"

# ============================================================================
# MODEL CONFIGURATION
# ============================================================================

CLASSES = ['AD', 'CN', 'EMCI', 'LMCI', 'MCI']
NUM_CLASSES = len(CLASSES)
CLASS_TO_IDX = {cls: idx for idx, cls in enumerate(CLASSES)}
ORIENTATIONS = ['axial', 'coronal', 'sagittal']

IMAGE_SIZE = 224
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

print("✅ Configuration loaded")
print(f"📊 Classes: {CLASSES}")
print(f"🖼️  Image size: {IMAGE_SIZE}×{IMAGE_SIZE}")

## 🏗️ Model Architecture Definitions

**Note**: We need to redefine the model classes to load the checkpoint

In [ ]:
# ============================================================================
# PHASE 2 CNN FEATURE EXTRACTOR (Frozen)
# ============================================================================

class ImprovedResNet50(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()
        self.backbone = models.resnet50(pretrained=True)
        
        for param in list(self.backbone.parameters())[:-60]:
            param.requires_grad = False
        
        num_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Sequential(
            nn.Dropout(0.6),
            nn.Linear(num_features, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )
    
    def forward(self, x):
        return self.backbone(x)


class FeatureExtractor(nn.Module):
    def __init__(self, original_model):
        super().__init__()
        self.features = nn.Sequential(*list(original_model.backbone.children())[:-1])
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
    
    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        return x


print("✅ Phase 2 CNN classes defined")

In [ ]:
# ============================================================================
# PHASE 3 ADVANCED MODEL WITH CROSS-ATTENTION
# ============================================================================

class CrossAttention(nn.Module):
    """Cross-attention module for fusing multi-orientation DeiT features"""
    
    def __init__(self, dim=384, num_heads=8, dropout=0.1):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.scale = self.head_dim ** -0.5
        
        self.q_proj = nn.Linear(dim, dim)
        self.k_proj = nn.Linear(dim, dim)
        self.v_proj = nn.Linear(dim, dim)
        
        self.attn_drop = nn.Dropout(dropout)
        self.proj = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(dropout)
    
    def forward(self, query, key, value):
        B = query.shape[0]
        
        q = self.q_proj(query).reshape(B, self.num_heads, self.head_dim)
        k = self.k_proj(key).reshape(B, self.num_heads, self.head_dim)
        v = self.v_proj(value).reshape(B, self.num_heads, self.head_dim)
        
        attn = (q * k).sum(dim=-1, keepdim=True) * self.scale
        attn = attn.softmax(dim=1)
        attn = self.attn_drop(attn)
        
        out = (attn * v).reshape(B, -1)
        
        out = self.proj(out)
        out = self.proj_drop(out)
        
        return out


class MultiOrientationCrossAttention(nn.Module):
    """Full cross-attention between all 3 orientations"""
    
    def __init__(self, dim=384, num_heads=8, dropout=0.1):
        super().__init__()
        
        self.axial_to_others = CrossAttention(dim, num_heads, dropout)
        self.coronal_to_others = CrossAttention(dim, num_heads, dropout)
        self.sagittal_to_others = CrossAttention(dim, num_heads, dropout)
        
        self.norm_axial = nn.LayerNorm(dim)
        self.norm_coronal = nn.LayerNorm(dim)
        self.norm_sagittal = nn.LayerNorm(dim)
        
        self.ffn = nn.Sequential(
            nn.Linear(dim * 3, dim * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim * 2, dim * 3)
        )
        self.norm_ffn = nn.LayerNorm(dim * 3)
    
    def forward(self, axial, coronal, sagittal):
        axial_attended = self.axial_to_others(
            axial, 
            torch.stack([coronal, sagittal], dim=1).mean(dim=1),
            torch.stack([coronal, sagittal], dim=1).mean(dim=1)
        )
        axial = self.norm_axial(axial + axial_attended)
        
        coronal_attended = self.coronal_to_others(
            coronal,
            torch.stack([axial, sagittal], dim=1).mean(dim=1),
            torch.stack([axial, sagittal], dim=1).mean(dim=1)
        )
        coronal = self.norm_coronal(coronal + coronal_attended)
        
        sagittal_attended = self.sagittal_to_others(
            sagittal,
            torch.stack([axial, coronal], dim=1).mean(dim=1),
            torch.stack([axial, coronal], dim=1).mean(dim=1)
        )
        sagittal = self.norm_sagittal(sagittal + sagittal_attended)
        
        fused = torch.cat([axial, coronal, sagittal], dim=1)
        fused = self.norm_ffn(fused + self.ffn(fused))
        
        return fused


class AdvancedCNNDeiTHybrid(nn.Module):
    """
    ADVANCED Hybrid model with:
    - DeiT-Small-Distilled (22M params)
    - Cross-attention between orientations
    - Enhanced fusion with gating
    """
    
    def __init__(self, num_classes=NUM_CLASSES, vit_model='deit_small_distilled_patch16_224'):
        super().__init__()
        
        self.vit = timm.create_model(vit_model, pretrained=True, num_classes=0)
        self.vit_feature_dim = self.vit.num_features  # 384
        
        for param in self.vit.parameters():
            param.requires_grad = True
        
        self.cnn_feature_dim = 2048 * 3  # 6144
        
        self.cross_attention = MultiOrientationCrossAttention(
            dim=self.vit_feature_dim,
            num_heads=8,
            dropout=0.2
        )
        
        self.vit_total_dim = self.vit_feature_dim * 3  # 1152
        
        self.feature_gate = nn.Sequential(
            nn.Linear(self.cnn_feature_dim + self.vit_total_dim, 1024),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(1024, 2),
            nn.Sigmoid()
        )
        
        self.fusion = nn.Sequential(
            nn.Linear(self.cnn_feature_dim + self.vit_total_dim, 2048),
            nn.ReLU(),
            nn.BatchNorm1d(2048),
            nn.Dropout(0.5),
            
            nn.Linear(2048, 1024),
            nn.ReLU(),
            nn.BatchNorm1d(1024),
            nn.Dropout(0.4),
            
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Dropout(0.3)
        )
        
        self.classifier = nn.Linear(512, num_classes)
    
    def forward(self, cnn_features, axial_img, coronal_img, sagittal_img):
        vit_axial = self.vit(axial_img)
        vit_coronal = self.vit(coronal_img)
        vit_sagittal = self.vit(sagittal_img)
        
        vit_features = self.cross_attention(vit_axial, vit_coronal, vit_sagittal)
        
        combined = torch.cat([cnn_features, vit_features], dim=1)
        
        gates = self.feature_gate(combined)
        cnn_gate = gates[:, 0:1]
        vit_gate = gates[:, 1:2]
        
        gated_cnn = cnn_features * cnn_gate
        gated_vit = vit_features * vit_gate
        combined_gated = torch.cat([gated_cnn, gated_vit], dim=1)
        
        fused = self.fusion(combined_gated)
        logits = self.classifier(fused)
        
        return logits


print("✅ Phase 3 ADVANCED model class defined")

## 📥 Load Trained Models

Loading:
1. **Phase 2 CNN Feature Extractors** (frozen, for multi-view features)
2. **Phase 3 ADVANCED Model** (trained DeiT with cross-attention)

In [ ]:
# Load frozen CNN feature extractors
print("🔧 Loading Phase 2 CNN feature extractors...\n")

feature_extractors = {}

for orientation in ORIENTATIONS:
    extractor_path = os.path.join(PHASE2_OUTPUT, f'{orientation}_feature_extractor.pth')
    checkpoint = torch.load(extractor_path, weights_only=False)
    
    base_model = ImprovedResNet50()
    extractor = FeatureExtractor(base_model)
    extractor.load_state_dict(checkpoint['state_dict'])
    extractor.to(device)
    extractor.eval()
    
    for param in extractor.parameters():
        param.requires_grad = False
    
    feature_extractors[orientation] = extractor
    
    print(f"✅ {orientation.capitalize()}: {checkpoint['test_accuracy']*100:.2f}% accuracy (frozen)")

print(f"\n✅ All CNN extractors loaded!")

In [ ]:
# Load Phase 3 ADVANCED trained model
print("🎯 Loading Phase 3 ADVANCED trained model...\n")

model = AdvancedCNNDeiTHybrid(num_classes=NUM_CLASSES).to(device)

checkpoint = torch.load(MODEL_CHECKPOINT, weights_only=False, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print(f"✅ Model loaded successfully!")
print(f"   Trained Epoch: {checkpoint['epoch'] + 1}")
print(f"   Validation Accuracy: {checkpoint['val_acc']:.2f}%")
print(f"   Validation Loss: {checkpoint['val_loss']:.4f}")

total_params = sum(p.numel() for p in model.parameters())
print(f"\n📊 Model Info:")
print(f"   Total parameters: {total_params:,}")
print(f"   Architecture: CNN-DeiT Hybrid with Cross-Attention")
print(f"   DeiT Model: deit_small_distilled_patch16_224 (22M params)")

## 🖼️ Image Preprocessing

Define transforms for test images (same as training)

In [ ]:
# Image preprocessing
test_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])

def load_image(image_path):
    """Load and preprocess a single image"""
    image = Image.open(image_path).convert('RGB')
    return test_transform(image).unsqueeze(0)  # Add batch dimension

print("✅ Image preprocessing ready")

## 🎲 Select a Random Test Subject

Load test data and randomly pick a subject with all 3 orientations

In [ ]:
# Load test dataset
test_df = pd.read_csv(TEST_CSV)

print(f"📊 Test Dataset:")
print(f"   Subjects: {len(test_df['subject_id'].unique())}")
print(f"   Total images: {len(test_df)}")
print()

# Randomly select a test subject
np.random.seed(42)  # For reproducibility (can change this)
test_subjects = test_df['subject_id'].unique()
random_subject = np.random.choice(test_subjects)

print(f"🎲 Randomly selected subject: {random_subject}")

# Get all images for this subject
subject_data = test_df[test_df['subject_id'] == random_subject]
print(f"\n📂 Subject images:")
print(subject_data[['subject_id', 'label', 'orientation', 'image_path']])

# Extract true label
true_label = subject_data.iloc[0]['label']
print(f"\n🏷️  True Label: {true_label}")

## 🔍 Load Multi-Orientation MRI Images

Load axial, coronal, and sagittal views for the selected subject

In [ ]:
# Load all 3 orientation images
images = {}
image_paths = {}

for _, row in subject_data.iterrows():
    orientation = row['orientation']
    image_path = os.path.join(DATA_PATH, row['image_path'])
    
    images[orientation] = load_image(image_path).to(device)
    image_paths[orientation] = image_path
    
    print(f"✅ Loaded {orientation}: {image_path}")

# Verify all orientations present
if len(images) != 3:
    print("⚠️  Warning: Not all 3 orientations found!")
else:
    print(f"\n✅ All 3 orientations loaded for subject {random_subject}")

## 📊 Visualize Input Images

Show the 3 MRI orientations that will be fed to the model

In [ ]:
# Visualize input images
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for idx, orientation in enumerate(ORIENTATIONS):
    # Convert tensor back to displayable image
    img_tensor = images[orientation][0].cpu()
    
    # Denormalize
    for t, m, s in zip(img_tensor, IMAGENET_MEAN, IMAGENET_STD):
        t.mul_(s).add_(m)
    
    img_np = img_tensor.permute(1, 2, 0).numpy()
    img_np = np.clip(img_np, 0, 1)
    
    axes[idx].imshow(img_np)
    axes[idx].set_title(f'{orientation.capitalize()} View', fontsize=14, fontweight='bold')
    axes[idx].axis('off')

fig.suptitle(f'Subject: {random_subject} | True Label: {true_label}', 
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("✅ Input images displayed")

## 🚀 Run Inference - Get All Predictions

Extract features and run the model to get predictions

In [ ]:
# ============================================================================
# INFERENCE: Extract features and predict
# ============================================================================

print("🔬 Running inference...\n")

with torch.no_grad():
    # Extract frozen CNN features from all 3 orientations
    axial_cnn = feature_extractors['axial'](images['axial'])
    coronal_cnn = feature_extractors['coronal'](images['coronal'])
    sagittal_cnn = feature_extractors['sagittal'](images['sagittal'])
    
    cnn_features = torch.cat([axial_cnn, coronal_cnn, sagittal_cnn], dim=1)
    
    print(f"✅ CNN Features extracted:")
    print(f"   Axial: {axial_cnn.shape}")
    print(f"   Coronal: {coronal_cnn.shape}")
    print(f"   Sagittal: {sagittal_cnn.shape}")
    print(f"   Combined: {cnn_features.shape}")
    print()
    
    # Run model
    logits = model(cnn_features, images['axial'], images['coronal'], images['sagittal'])
    
    # Get probabilities
    probabilities = F.softmax(logits, dim=1)
    
    # Get predicted class
    confidence, predicted_idx = logits.max(1)
    predicted_idx = predicted_idx.item()
    predicted_class = CLASSES[predicted_idx]
    
    print(f"✅ Model inference complete!")

print("\n" + "="*80)
print("PREDICTION RESULTS")
print("="*80)

## 📈 Display All Output Formats

Show logits, probabilities, predicted class, and confidence scores

In [ ]:
# ============================================================================
# DISPLAY ALL OUTPUT FORMATS
# ============================================================================

print("\n1️⃣  LOGITS (Raw Model Output):")
print(f"   Shape: {logits.shape}  # [batch_size=1, num_classes=5]")
print(f"   Values: {logits[0].cpu().numpy()}")
print(f"   ℹ️  Unnormalized scores (used for loss calculation)")
print()

print("2️⃣  PROBABILITIES (After Softmax):")
print(f"   Shape: {probabilities.shape}")
print(f"   Values: {probabilities[0].cpu().numpy()}")
print(f"   Sum: {probabilities[0].sum().item():.6f}  # Should be 1.0")
print()

print("3️⃣  CLASS-WISE CONFIDENCE SCORES:")
print()
for i, cls in enumerate(CLASSES):
    prob = probabilities[0, i].item()
    bar_length = int(prob * 50)
    bar = '█' * bar_length + '░' * (50 - bar_length)
    
    color = '🟢' if cls == predicted_class else '⚪'
    marker = ' ← PREDICTED' if cls == predicted_class else ''
    true_marker = ' ← TRUE LABEL' if cls == true_label else ''
    
    print(f"{color} {cls:5s}: {prob*100:6.2f}%  {bar}{marker}{true_marker}")

print()
print("4️⃣  PREDICTED CLASS:")
print(f"   Index: {predicted_idx}")
print(f"   Name: '{predicted_class}'")
print(f"   Confidence: {probabilities[0, predicted_idx].item()*100:.2f}%")
print()

print("5️⃣  TOP-3 PREDICTIONS:")
top3_probs, top3_indices = torch.topk(probabilities[0], 3)
for rank, (idx, prob) in enumerate(zip(top3_indices, top3_probs), 1):
    print(f"   {rank}. {CLASSES[idx.item()]:5s}: {prob.item()*100:5.2f}%")

print()
print("="*80)
print("🎯 FINAL PREDICTION")
print("="*80)
print(f"   Predicted: {predicted_class} ({probabilities[0, predicted_idx].item()*100:.1f}% confidence)")
print(f"   True Label: {true_label}")
print(f"   Result: {'✅ CORRECT!' if predicted_class == true_label else '❌ INCORRECT'}")
print("="*80)

## 📊 Visualize Prediction Results

Create comprehensive visualization of the prediction

In [ ]:
# ============================================================================
# COMPREHENSIVE VISUALIZATION
# ============================================================================

fig = plt.figure(figsize=(18, 10))
gs = fig.add_gridspec(2, 3, hspace=0.3, wspace=0.3)

# Row 1: MRI Images
for idx, orientation in enumerate(ORIENTATIONS):
    ax = fig.add_subplot(gs[0, idx])
    
    # Denormalize image
    img_tensor = images[orientation][0].cpu()
    for t, m, s in zip(img_tensor, IMAGENET_MEAN, IMAGENET_STD):
        t.mul_(s).add_(m)
    img_np = img_tensor.permute(1, 2, 0).numpy()
    img_np = np.clip(img_np, 0, 1)
    
    ax.imshow(img_np)
    ax.set_title(f'{orientation.capitalize()} View', fontsize=13, fontweight='bold')
    ax.axis('off')

# Row 2 Left: Probability Bar Chart
ax_bar = fig.add_subplot(gs[1, 0:2])
probs_np = probabilities[0].cpu().numpy() * 100

colors = ['#2ecc71' if CLASSES[i] == predicted_class else '#95a5a6' for i in range(NUM_CLASSES)]
bars = ax_bar.barh(CLASSES, probs_np, color=colors, edgecolor='black', linewidth=2)

# Highlight true label
true_idx = CLASS_TO_IDX[true_label]
bars[true_idx].set_edgecolor('red')
bars[true_idx].set_linewidth(3)

ax_bar.set_xlabel('Confidence (%)', fontsize=12, fontweight='bold')
ax_bar.set_title('Prediction Confidence Scores', fontsize=14, fontweight='bold')
ax_bar.set_xlim(0, 100)
ax_bar.grid(axis='x', alpha=0.3)

# Add value labels
for i, (cls, prob) in enumerate(zip(CLASSES, probs_np)):
    ax_bar.text(prob + 2, i, f'{prob:.1f}%', va='center', fontsize=11, fontweight='bold')

# Row 2 Right: Prediction Summary
ax_text = fig.add_subplot(gs[1, 2])
ax_text.axis('off')

summary_text = f"""
━━━━━━━━━━━━━━━━━━━━━
  PREDICTION SUMMARY
━━━━━━━━━━━━━━━━━━━━━

Subject ID:
  {random_subject}

True Label:
  {true_label}

Predicted:
  {predicted_class}

Confidence:
  {probabilities[0, predicted_idx].item()*100:.1f}%

Result:
  {'✅ CORRECT' if predicted_class == true_label else '❌ INCORRECT'}

━━━━━━━━━━━━━━━━━━━━━
TOP-3 PREDICTIONS:

1. {CLASSES[top3_indices[0].item()]}: {top3_probs[0].item()*100:.1f}%
2. {CLASSES[top3_indices[1].item()]}: {top3_probs[1].item()*100:.1f}%
3. {CLASSES[top3_indices[2].item()]}: {top3_probs[2].item()*100:.1f}%

━━━━━━━━━━━━━━━━━━━━━
Model: Phase 3 ADVANCED
CNN-DeiT Cross-Attention
━━━━━━━━━━━━━━━━━━━━━
"""

ax_text.text(0.5, 0.5, summary_text, 
             ha='center', va='center',
             fontsize=11, 
             family='monospace',
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.3))

# Overall title
fig.suptitle(f'Alzheimer\'s Disease Classification - Subject {random_subject}', 
             fontsize=16, fontweight='bold', y=0.98)

plt.tight_layout()
plt.show()

print("✅ Comprehensive visualization complete!")

## 🎯 Model Architecture Visualization

Show what happens inside the model during inference

In [ ]:
# Architecture flow diagram
fig, ax = plt.subplots(figsize=(14, 10))
ax.axis('off')

architecture_diagram = """
╔═══════════════════════════════════════════════════════════════════════════╗
║                    INFERENCE PIPELINE ARCHITECTURE                        ║
╚═══════════════════════════════════════════════════════════════════════════╝

┌─────────────┐      ┌─────────────┐      ┌─────────────┐
│   AXIAL     │      │  CORONAL    │      │  SAGITTAL   │
│  224×224    │      │  224×224    │      │  224×224    │
└──────┬──────┘      └──────┬──────┘      └──────┬──────┘
       │                    │                     │
       ▼                    ▼                     ▼
┌─────────────┐      ┌─────────────┐      ┌─────────────┐
│  ResNet-50  │      │  ResNet-50  │      │  ResNet-50  │
│   (FROZEN)  │      │   (FROZEN)  │      │   (FROZEN)  │
└──────┬──────┘      └──────┬──────┘      └──────┬──────┘
       │                    │                     │
       │   [2048-D]         │   [2048-D]          │   [2048-D]
       │                    │                     │
       └────────────────────┼─────────────────────┘
                            │
                            ▼
                   ┌─────────────────┐
                   │  CNN Features   │
                   │    (6144-D)     │
                   └────────┬────────┘
                            │
       ┌────────────────────┼────────────────────┐
       │                    │                    │
       ▼                    ▼                    ▼
┌─────────────┐      ┌─────────────┐      ┌─────────────┐
│    DeiT     │      │    DeiT     │      │    DeiT     │
│   Axial     │      │  Coronal    │      │  Sagittal   │
│  (22M params)│     │ (SHARED)    │      │  (SHARED)   │
└──────┬──────┘      └──────┬──────┘      └──────┬──────┘
       │                    │                     │
       │   [384-D]          │   [384-D]           │   [384-D]
       │                    │                     │
       └────────────────────┼─────────────────────┘
                            │
                            ▼
                ┌───────────────────────┐
                │  CROSS-ATTENTION      │
                │  (Multi-Head, 8 heads)│
                │  Orientations Interact│
                └───────────┬───────────┘
                            │
                            │   [1152-D]
                            ▼
               ┌─────────────────────────┐
               │   ADAPTIVE GATING       │
               │  (Learn CNN vs DeiT wt) │
               └───────────┬─────────────┘
                           │
                           │   [7296-D]
                           ▼
               ┌─────────────────────────┐
               │   FUSION MLP            │
               │  7296→2048→1024→512     │
               └───────────┬─────────────┘
                           │
                           │   [512-D]
                           ▼
               ┌─────────────────────────┐
               │    CLASSIFIER           │
               │    (Linear Layer)       │
               └───────────┬─────────────┘
                           │
                           │   [5 classes]
                           ▼
            ┌────────────────────────────────┐
            │  SOFTMAX (Probabilities)       │
            │  AD, CN, EMCI, LMCI, MCI       │
            └────────────────────────────────┘

KEY FEATURES:
✅ Multi-orientation processing (3 views)
✅ Frozen CNN features (transfer learning)
✅ DeiT with knowledge distillation (22M params)
✅ Cross-attention between orientations
✅ Adaptive feature gating
✅ Deep fusion network

TOTAL PARAMETERS: ~30M (trainable: ~25M)
"""

ax.text(0.5, 0.5, architecture_diagram,
        ha='center', va='center',
        fontsize=9.5,
        family='monospace',
        bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.2))

plt.title('Model Architecture Flow', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

print("✅ Architecture diagram displayed")

## 🔄 Try Another Random Subject

Run this cell to test on a different subject

In [ ]:
# Quick inference function for any subject
def predict_subject(subject_id):
    """Run inference on a specific subject"""
    
    # Get subject data
    subject_data = test_df[test_df['subject_id'] == subject_id]
    
    if len(subject_data) == 0:
        print(f"❌ Subject {subject_id} not found in test set!")
        return
    
    true_label = subject_data.iloc[0]['label']
    
    # Load images
    images = {}
    for _, row in subject_data.iterrows():
        orientation = row['orientation']
        image_path = os.path.join(DATA_PATH, row['image_path'])
        images[orientation] = load_image(image_path).to(device)
    
    # Run inference
    with torch.no_grad():
        axial_cnn = feature_extractors['axial'](images['axial'])
        coronal_cnn = feature_extractors['coronal'](images['coronal'])
        sagittal_cnn = feature_extractors['sagittal'](images['sagittal'])
        cnn_features = torch.cat([axial_cnn, coronal_cnn, sagittal_cnn], dim=1)
        
        logits = model(cnn_features, images['axial'], images['coronal'], images['sagittal'])
        probabilities = F.softmax(logits, dim=1)
        _, predicted_idx = logits.max(1)
        predicted_class = CLASSES[predicted_idx.item()]
    
    # Display results
    print(f"\n{'='*80}")
    print(f"PREDICTION FOR SUBJECT: {subject_id}")
    print(f"{'='*80}")
    print(f"True Label: {true_label}")
    print(f"Predicted: {predicted_class} ({probabilities[0, predicted_idx].item()*100:.1f}% confidence)")
    print(f"Result: {'✅ CORRECT!' if predicted_class == true_label else '❌ INCORRECT'}")
    print(f"\nAll Probabilities:")
    for i, cls in enumerate(CLASSES):
        prob = probabilities[0, i].item()
        bar = '█' * int(prob * 30)
        print(f"  {cls:5s}: {prob*100:5.1f}%  {bar}")
    print(f"{'='*80}\n")

# Try another random subject
another_subject = np.random.choice(test_subjects)
print(f"🎲 Testing another subject: {another_subject}")
predict_subject(another_subject)

# You can also manually specify a subject
# predict_subject("002_S_0295")  # Replace with any test subject ID

## 💾 Save Prediction Report

Generate a text report for documentation

In [ ]:
# Generate prediction report
report = f"""
{'='*80}
ALZHEIMER'S DISEASE CLASSIFICATION - INFERENCE REPORT
{'='*80}

Date: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}
Model: Phase 3 ADVANCED - CNN-DeiT Hybrid with Cross-Attention

{'='*80}
SUBJECT INFORMATION
{'='*80}
Subject ID: {random_subject}
True Diagnosis: {true_label}

{'='*80}
MODEL PREDICTION
{'='*80}
Predicted Class: {predicted_class}
Confidence: {probabilities[0, predicted_idx].item()*100:.2f}%
Prediction Result: {'CORRECT ✓' if predicted_class == true_label else 'INCORRECT ✗'}

{'='*80}
CONFIDENCE SCORES FOR ALL CLASSES
{'='*80}
"""

for i, cls in enumerate(CLASSES):
    prob = probabilities[0, i].item()
    report += f"{cls:10s}: {prob*100:6.2f}%\n"

report += f"""
{'='*80}
TOP-3 PREDICTIONS
{'='*80}
"""

for rank, (idx, prob) in enumerate(zip(top3_indices, top3_probs), 1):
    report += f"{rank}. {CLASSES[idx.item()]:5s}: {prob.item()*100:6.2f}%\n"

report += f"""
{'='*80}
MODEL ARCHITECTURE
{'='*80}
CNN Branch: Multi-orientation ResNet-50 (frozen)
  - Axial features: 2048-D
  - Coronal features: 2048-D
  - Sagittal features: 2048-D
  - Total CNN features: 6144-D

DeiT Branch: deit_small_distilled_patch16_224
  - Parameters: 22M
  - Feature dim per orientation: 384-D
  - Cross-attended features: 1152-D

Fusion Network: 7296-D → 2048-D → 1024-D → 512-D → 5 classes

Key Innovations:
  ✓ Multi-orientation cross-attention (8 heads)
  ✓ Knowledge distillation (DeiT)
  ✓ Adaptive feature gating
  ✓ Deep fusion architecture

{'='*80}
CLASSIFICATION LEGEND
{'='*80}
AD   : Alzheimer's Disease (severe cognitive impairment)
CN   : Cognitively Normal (healthy control)
EMCI : Early Mild Cognitive Impairment
LMCI : Late Mild Cognitive Impairment
MCI  : Mild Cognitive Impairment

{'='*80}
END OF REPORT
{'='*80}
"""

# Print report
print(report)

# Optionally save to file
# with open(f'prediction_report_{random_subject}.txt', 'w') as f:
#     f.write(report)
# print(f"✅ Report saved to: prediction_report_{random_subject}.txt")

## 🎓 For Thesis Viva/Defense

**How to use this notebook during your viva:**

1. **Quick Demo** (2-3 minutes):
   - Run cells 1-14 sequentially
   - Shows complete prediction pipeline
   - Impressive visualizations for reviewers

2. **Answer Questions**:
   - "How does inference work?" → Show architecture diagram
   - "What's the output format?" → Point to probability scores
   - "Can you test another image?" → Run the "Try Another Subject" cell
   - "What's the confidence?" → Show probability bar chart

3. **Key Points to Highlight**:
   - ✅ Multi-orientation processing (axial, coronal, sagittal)
   - ✅ Transfer learning (frozen CNN + trainable DeiT)
   - ✅ Cross-attention mechanism (novel contribution)
   - ✅ Knowledge distillation (DeiT for limited medical data)
   - ✅ Fast inference (~1-2 seconds per subject)

4. **Technical Advantages**:
   - No retraining needed
   - Works on any test subject
   - Comprehensive output formats
   - Production-ready architecture

**Good luck with your defense! 🎓**